
# Урок 6 — Метрики, пороги и бизнес-стоимость ошибок

**Цель:** связать метрики модели с бизнес-решениями.  
**Что делаем:** генерируем синтетический churn-кейс, строим ROC/PR, считаем метрики при разных порогах и подбираем порог по **стоимости ошибок** (FP/FN).


In [ ]:
# Импорты
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay, confusion_matrix
)

np.random.seed(42)
print("Импорты готовы")


## 1) Синтетические данные
- `y_true` — истинный churn (1 = ушёл), класс несбалансирован (около 20%).
- `y_proba` — вероятности churn от модели (имитируем адекватную модель).


In [ ]:
n = 500
y_true = np.random.choice([0, 1], size=n, p=[0.8, 0.2])  # 20% оттока
y_proba = np.clip(np.random.normal(0.3 + 0.4 * y_true, 0.15, size=n), 0, 1)

df = pd.DataFrame({"y_true": y_true, "y_proba": y_proba})
df.head()


## 2) Базовые метрики при пороге 0.5


In [ ]:
threshold = 0.5
y_pred = (df["y_proba"] >= threshold).astype(int)

acc = accuracy_score(df["y_true"], y_pred)
prec = precision_score(df["y_true"], y_pred, zero_division=0)
rec = recall_score(df["y_true"], y_pred)
f1 = f1_score(df["y_true"], y_pred, zero_division=0)
roc = roc_auc_score(df["y_true"], df["y_proba"])

print(f"Accuracy: {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall: {rec:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"ROC-AUC: {roc:.3f}")


## 3) ROC и Precision–Recall кривые


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
RocCurveDisplay.from_predictions(df["y_true"], df["y_proba"], ax=axes[0])
axes[0].set_title("ROC-кривая")

PrecisionRecallDisplay.from_predictions(df["y_true"], df["y_proba"], ax=axes[1])
axes[1].set_title("Precision–Recall кривая")
plt.tight_layout()
plt.show()


## 4) Метрики при разных порогах


In [ ]:
thresholds = np.linspace(0.1, 0.9, 9)
rows = []
for t in thresholds:
    y_pred_t = (df["y_proba"] >= t).astype(int)
    rows.append({
        "threshold": t,
        "precision": precision_score(df["y_true"], y_pred_t, zero_division=0),
        "recall": recall_score(df["y_true"], y_pred_t),
        "f1": f1_score(df["y_true"], y_pred_t, zero_division=0),
        "accuracy": accuracy_score(df["y_true"], y_pred_t)
    })
met_df = pd.DataFrame(rows)
met_df


## 5) Бизнес-стоимость ошибок и подбор порога
- FP (ложная тревога): потратили на удержание клиента, который **и так не ушёл**.  
- FN (пропустили уход): потеряли маржу/LTV клиента, который **ушёл**.

*Задайте свои стоимости ниже.*


In [ ]:
FP_cost = 200   # руб., стоимость "лишней" удерживающей активности
FN_cost = 3000  # руб., стоимость упущенного уходящего клиента

cost_rows = []
for t in thresholds:
    y_pred_t = (df["y_proba"] >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(df["y_true"], y_pred_t).ravel()
    total_cost = fp * FP_cost + fn * FN_cost
    cost_rows.append({"threshold": t, "FP": fp, "FN": fn, "total_cost": total_cost})

cost_df = pd.DataFrame(cost_rows)
cost_df


## 6) Стоимость vs порог (поиск оптимума)


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(cost_df["threshold"], cost_df["total_cost"], marker="o")
plt.title("Стоимость ошибок при разных порогах")
plt.xlabel("Порог вероятности")
plt.ylabel("Суммарная стоимость (руб.)")
plt.grid(True)
plt.show()

best_idx = cost_df["total_cost"].idxmin()
best_t = cost_df.loc[best_idx, "threshold"]
best_cost = cost_df.loc[best_idx, "total_cost"]
print(f"📉 Минимальные потери при пороге ≈ {best_t:.2f} (стоимость ≈ {best_cost:.0f} руб.)")


## 7) Вопросы для обсуждения
1) Как менялись precision/recall при росте порога?  
2) Что важнее бизнесу в churn: минимизировать FN (не пропустить уходящих) или FP (не тратить впустую)?  
3) Как изменится оптимальный порог, если `FN_cost` в 10 раз больше `FP_cost`?  
4) Как переносить эту логику в CRM: сегменты, лимиты, A/B порогов?
